# Stage 1 — Data Acquisition & Exploratory Data Analysis (EDA)

**Case study:** Categorizing Trends in Science (arXiv metadata)

### Why this stage matters
Before touching any ML algorithm, we need to understand what we're working with: how big is the dataset, what fields does it have, are there missing values, and — critically for a 2M+ row JSON file — **can we even load all of it into memory?** (Usually not, on a laptop.) This notebook documents the decisions made here so they can be defended later.

### The dataset
The arXiv metadata snapshot is a single large JSON-Lines file (`arxiv-metadata-oai-snapshot.json`), one JSON object per line, one line per paper. Fields include: `id`, `title`, `abstract`, `categories`, `authors`, `update_date`, etc.

### Sampling strategy (documented decision — you should justify your own choice here)
Loading 2M+ full abstracts is unnecessary and slow for a case study. A defensible strategy:
- Filter to a small number of **categories** you're interested in (e.g. `cs.LG`, `cs.CL`, `cs.CV` — machine learning / NLP / computer vision), AND/OR
- Filter to a **recent date range** (e.g. last 2-3 years) to focus on *current* trends, which is literally what the task asks for.

This keeps the dataset to tens of thousands of rows — big enough to be meaningful, small enough to run TF-IDF + clustering on a laptop without special infrastructure.

In [16]:
import json
import pandas as pd

RAW_PATH = "../data/raw/arxiv-metadata-oai-snapshot.json"

# Categories to keep -- adjust to your interest area.
# See https://arxiv.org/category_taxonomy for the full list of category codes.
CATEGORIES_OF_INTEREST = {"cs.LG", "cs.CL", "cs.CV", "cs.AI"}

# Only keep papers updated from this year onward (keeps the analysis about *current* trends)
MIN_YEAR = 2024


In [ ]:
def paper_matches_filter(paper: dict) -> bool:
    """Return True if this paper should be kept, based on category and date filters."""
    cats = set(paper.get("categories", "").split())
    if not cats & CATEGORIES_OF_INTEREST:
        return False
    update_date = paper.get("update_date", "")
    if not update_date or int(update_date[:4]) < MIN_YEAR:
        return False
    return True


def load_filtered_arxiv(path: str, max_rows: int | None = None) -> pd.DataFrame:
    """Stream the large JSON-Lines file and keep only rows matching our filter.

    We stream line-by-line (rather than json.load on the whole file) because the
    file is too large to fit comfortably in memory as one JSON structure.
    """
    records = []
    with open(path, "r") as f:
        for line in f:
            paper = json.loads(line)
            if paper_matches_filter(paper):
                records.append({
                    "id": paper.get("id"),
                    "title": paper.get("title", "").replace("\n", " ").strip(),
                    "abstract": paper.get("abstract", "").replace("\n", " ").strip(),
                    "categories": paper.get("categories", ""),
                    "update_date": paper.get("update_date"),
                })
            if max_rows and len(records) >= max_rows:
                break
    return pd.DataFrame(records)


df = load_filtered_arxiv(RAW_PATH)
print("Rows after filtering:", df.shape)

SAMPLE_SIZE = 30000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

df.to_parquet("../data/processed/arxiv_filtered.parquet")
df.shape

## EDA checklist
Once `df` is loaded, walk through these — each answers a real question about the data:

1. **Shape & schema** — `df.shape`, `df.dtypes`, `df.head()`
2. **Missing values** — `df.isna().sum()` — any empty abstracts/titles to drop?
3. **Category distribution** — bar chart of `categories` (a paper can have multiple, so you may want to explode this column)
4. **Publications over time** — line chart of paper counts per month/year, per category — motivates the "trends" framing
5. **Abstract length distribution** — histogram of word counts — informs later TF-IDF parameter choices (e.g. `max_df`, `min_df`)
6. **Duplicate check** — `df.duplicated(subset='id').sum()`

Write 2-3 sentences after each plot: what do you see, and what decision does it inform for the next stage?

In [4]:
# TODO: EDA cells go here — shape, missingness, category counts, time trend, abstract length histogram
